# Gold FK/PK Constraints

Applies informational PK and FK constraints to gold layer Delta tables.


In [ ]:
# Gold FK/PK Constraints (Idempotent)
# Applies informational PK and FK constraints to gold layer Delta tables.

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gold_fk_pk_constraints")


def table_exists(full_name: str) -> bool:
    return spark.catalog.tableExists(full_name)


def get_create_stmt(full_name: str) -> str:
    rows = spark.sql(f"SHOW CREATE TABLE {full_name}").collect()
    return "\n".join([row[0] for row in rows]) if rows else ""


def is_nullable(full_name: str, column_name: str) -> bool:
    schema = spark.table(full_name).schema
    for field in schema.fields:
        if field.name.lower() == column_name.lower():
            return field.nullable
    raise AssertionError(f"Column {column_name} not found in {full_name}")


def set_not_null_if_needed(full_name: str, column_name: str):
    if not table_exists(full_name):
        logger.warning(f"Skipping NOT NULL: table missing {full_name}")
        return
    if is_nullable(full_name, column_name):
        logger.info(f"Setting NOT NULL: {full_name}.{column_name}")
        spark.sql(f"ALTER TABLE {full_name} ALTER COLUMN {column_name} SET NOT NULL")
    else:
        logger.info(f"NOT NULL already set: {full_name}.{column_name}")


def add_constraint_if_missing(full_name: str, constraint_name: str, ddl: str):
    logger.info(f'processing: {full_name}, {constraint_name}, {ddl}')
    if not table_exists(full_name):
        logger.warning(f"Skipping constraint {constraint_name}: table missing {full_name}")
        return
    stmt = get_create_stmt(full_name).lower()
    if constraint_name.lower() in stmt:
        logger.info(f"Constraint exists: {constraint_name} on {full_name}")
        return
    logger.info(f"Adding constraint: {constraint_name} on {full_name}")
    spark.sql(ddl)


# Ensure PK columns are NOT NULL (required for PRIMARY KEY)
not_null_columns = [
    ("wheelie.gold.dim_date", "date_key"),
    ("wheelie.gold.dim_service_date", "service_date_key"),
    ("wheelie.gold.dim_rental_date", "rental_date_key"),
    ("wheelie.gold.dim_return_date", "return_date_key"),
    ("wheelie.gold.dim_payment_date", "payment_date_key"),
    ("wheelie.gold.dim_payment_deadline_date", "payment_deadline_date_key"),
    ("wheelie.gold.dim_staff", "staff_key"),
    ("wheelie.gold.dim_manager", "manager_key"),
    ("wheelie.gold.dim_store", "store_key"),
    ("wheelie.gold.dim_car", "car_key"),
    ("wheelie.gold.dim_customer", "customer_key"),
    ("wheelie.gold.dim_equipment", "equipment_key"),
    ("wheelie.gold.fact_service", "service_key"),
    ("wheelie.gold.fact_service", "car_key"),
    ("wheelie.gold.fact_rental", "rental_key"),
    ("wheelie.gold.fact_rental", "customer_key"),
    ("wheelie.gold.fact_rental", "car_key"),
    ("wheelie.gold.fact_rental", "staff_key"),
    ("wheelie.gold.fact_rental", "store_key"),
    ("wheelie.gold.fact_rental", "rental_date_key"),
    ("wheelie.gold.fact_rental", "payment_deadline_date_key"),
    ("wheelie.gold.bridge_staff_hierarchy", "staff_key"),
    ("wheelie.gold.bridge_staff_hierarchy", "manager_key"),
    ("wheelie.gold.bridge_staff_hierarchy", "level"),
    ("wheelie.gold.bridge_car_equipment", "car_key"),
    ("wheelie.gold.bridge_equipment_group_equipment", "equipment_group_key"),
    ("wheelie.gold.bridge_equipment_group_equipment", "equipment_key"),
]

for full_name, column_name in not_null_columns:
    set_not_null_if_needed(full_name, column_name)


# Primary keys (informational only)
constraints = [
    (
        "wheelie.gold.dim_date",
        "pk_dim_date",
        "ALTER TABLE wheelie.gold.dim_date ADD CONSTRAINT pk_dim_date PRIMARY KEY (date_key)",
    ),
    (
        "wheelie.gold.dim_service_date",
        "pk_dim_service_date",
        "ALTER TABLE wheelie.gold.dim_service_date ADD CONSTRAINT pk_dim_service_date PRIMARY KEY (service_date_key)",
    ),
    (
        "wheelie.gold.dim_rental_date",
        "pk_dim_rental_date",
        "ALTER TABLE wheelie.gold.dim_rental_date ADD CONSTRAINT pk_dim_rental_date PRIMARY KEY (rental_date_key)",
    ),
    (
        "wheelie.gold.dim_return_date",
        "pk_dim_return_date",
        "ALTER TABLE wheelie.gold.dim_return_date ADD CONSTRAINT pk_dim_return_date PRIMARY KEY (return_date_key)",
    ),
    (
        "wheelie.gold.dim_payment_date",
        "pk_dim_payment_date",
        "ALTER TABLE wheelie.gold.dim_payment_date ADD CONSTRAINT pk_dim_payment_date PRIMARY KEY (payment_date_key)",
    ),
    (
        "wheelie.gold.dim_payment_deadline_date",
        "pk_dim_payment_deadline_date",
        "ALTER TABLE wheelie.gold.dim_payment_deadline_date ADD CONSTRAINT pk_dim_payment_deadline_date PRIMARY KEY (payment_deadline_date_key)",
    ),
    (
        "wheelie.gold.dim_staff",
        "pk_dim_staff",
        "ALTER TABLE wheelie.gold.dim_staff ADD CONSTRAINT pk_dim_staff PRIMARY KEY (staff_key)",
    ),
    (
        "wheelie.gold.dim_manager",
        "pk_dim_manager",
        "ALTER TABLE wheelie.gold.dim_manager ADD CONSTRAINT pk_dim_manager PRIMARY KEY (manager_key)",
    ),
    (
        "wheelie.gold.dim_store",
        "pk_dim_store",
        "ALTER TABLE wheelie.gold.dim_store ADD CONSTRAINT pk_dim_store PRIMARY KEY (store_key)",
    ),
    (
        "wheelie.gold.dim_car",
        "pk_dim_car",
        "ALTER TABLE wheelie.gold.dim_car ADD CONSTRAINT pk_dim_car PRIMARY KEY (car_key)",
    ),
    (
        "wheelie.gold.dim_customer",
        "pk_dim_customer",
        "ALTER TABLE wheelie.gold.dim_customer ADD CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key)",
    ),
    (
        "wheelie.gold.dim_equipment",
        "pk_dim_equipment",
        "ALTER TABLE wheelie.gold.dim_equipment ADD CONSTRAINT pk_dim_equipment PRIMARY KEY (equipment_key)",
    ),
    (
        "wheelie.gold.fact_service",
        "pk_fact_service",
        "ALTER TABLE wheelie.gold.fact_service ADD CONSTRAINT pk_fact_service PRIMARY KEY (service_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "pk_fact_rental",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT pk_fact_rental PRIMARY KEY (rental_key)",
    ),
    (
        "wheelie.gold.bridge_staff_hierarchy",
        "pk_bridge_staff_hierarchy",
        "ALTER TABLE wheelie.gold.bridge_staff_hierarchy ADD CONSTRAINT pk_bridge_staff_hierarchy PRIMARY KEY (staff_key, manager_key, level)",
    ),
    (
        "wheelie.gold.bridge_car_equipment",
        "pk_bridge_car_equipment",
        "ALTER TABLE wheelie.gold.bridge_car_equipment ADD CONSTRAINT pk_bridge_car_equipment PRIMARY KEY (car_key)",
    ),
    (
        "wheelie.gold.bridge_equipment_group_equipment",
        "pk_bridge_equipment_group_equipment",
        "ALTER TABLE wheelie.gold.bridge_equipment_group_equipment ADD CONSTRAINT pk_bridge_equipment_group_equipment PRIMARY KEY (equipment_group_key, equipment_key)",
    ),
    # Foreign keys
    (
        "wheelie.gold.fact_service",
        "fk_fact_service_car",
        "ALTER TABLE wheelie.gold.fact_service ADD CONSTRAINT fk_fact_service_car FOREIGN KEY (car_key) REFERENCES wheelie.gold.dim_car (car_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_customer",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_customer FOREIGN KEY (customer_key) REFERENCES wheelie.gold.dim_customer (customer_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_car",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_car FOREIGN KEY (car_key) REFERENCES wheelie.gold.dim_car (car_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_staff",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_staff FOREIGN KEY (staff_key) REFERENCES wheelie.gold.dim_staff (staff_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_store",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_store FOREIGN KEY (store_key) REFERENCES wheelie.gold.dim_store (store_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_rental_date",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_rental_date FOREIGN KEY (rental_date_key) REFERENCES wheelie.gold.dim_rental_date (rental_date_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_return_date",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_return_date FOREIGN KEY (return_date_key) REFERENCES wheelie.gold.dim_return_date (return_date_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_payment_date",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_payment_date FOREIGN KEY (payment_date_key) REFERENCES wheelie.gold.dim_payment_date (payment_date_key)",
    ),
    (
        "wheelie.gold.fact_rental",
        "fk_fact_rental_payment_deadline_date",
        "ALTER TABLE wheelie.gold.fact_rental ADD CONSTRAINT fk_fact_rental_payment_deadline_date FOREIGN KEY (payment_deadline_date_key) REFERENCES wheelie.gold.dim_payment_deadline_date (payment_deadline_date_key)",
    ),
    (
        "wheelie.gold.bridge_staff_hierarchy",
        "fk_bridge_staff_staff",
        "ALTER TABLE wheelie.gold.bridge_staff_hierarchy ADD CONSTRAINT fk_bridge_staff_staff FOREIGN KEY (staff_key) REFERENCES wheelie.gold.dim_staff (staff_key)",
    ),
    (
        "wheelie.gold.bridge_staff_hierarchy",
        "fk_bridge_staff_manager",
        "ALTER TABLE wheelie.gold.bridge_staff_hierarchy ADD CONSTRAINT fk_bridge_staff_manager FOREIGN KEY (manager_key) REFERENCES wheelie.gold.dim_manager (manager_key)",
    ),
    (
        "wheelie.gold.bridge_car_equipment",
        "fk_bridge_car_equipment_car",
        "ALTER TABLE wheelie.gold.bridge_car_equipment ADD CONSTRAINT fk_bridge_car_equipment_car FOREIGN KEY (car_key) REFERENCES wheelie.gold.dim_car (car_key)",
    ),
    (
        "wheelie.gold.bridge_equipment_group_equipment",
        "fk_bridge_equipment_group_equipment_equipment",
        "ALTER TABLE wheelie.gold.bridge_equipment_group_equipment ADD CONSTRAINT fk_bridge_equipment_group_equipment_equipment FOREIGN KEY (equipment_key) REFERENCES wheelie.gold.dim_equipment (equipment_key)",
    ),
]

for full_name, constraint_name, ddl in constraints:
    add_constraint_if_missing(full_name, constraint_name, ddl)

logger.info("Gold FK/PK constraints applied (idempotent)")
